# Colab Two-Tower Baseline Training

Self-contained Colab notebook for the first serious Two-Tower MLP baseline.

It uses the Gold dataset already saved in Drive:

```text
/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab/
```

Silver data is not required for training. This notebook does not clone the GitHub repo and does not require MLflow; it saves metrics and artifacts directly to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

GOLD_ROOT = Path('/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab')
OUTPUT_DIR = Path('/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v1')
ALS_SUMMARY_PATH = Path('/content/drive/MyDrive/recsys/data/gold/als/v1_colab/evaluation_summary.json')

assert GOLD_ROOT.exists(), f'Missing Gold root: {GOLD_ROOT}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Gold root:', GOLD_ROOT)
print('Output dir:', OUTPUT_DIR)
print('ALS summary exists:', ALS_SUMMARY_PATH.exists())

In [ ]:
required_tables = {
    'train': GOLD_ROOT / 'train',
    'validation': GOLD_ROOT / 'validation',
    'targets_train': GOLD_ROOT / 'targets/train',
    'targets_validation': GOLD_ROOT / 'targets/validation',
    'user_state_train': GOLD_ROOT / 'user_state/train',
    'user_state_validation': GOLD_ROOT / 'user_state/validation',
    'item_pit_train': GOLD_ROOT / 'item_features/point_in_time/train',
    'item_pit_validation': GOLD_ROOT / 'item_features/point_in_time/validation',
    'vocabularies': GOLD_ROOT / 'vocabularies',
    'transforms': GOLD_ROOT / 'transforms',
}

print('Required table parquet file counts:')
for name, path in required_tables.items():
    if path.exists():
        count = len(list(path.rglob('*.parquet')))
        print(f'{name:24s} {count:4d} parquet files | {path}')
    else:
        print(f'{name:24s} MISSING | {path}')

## Dependency check

Colab usually already has `torch`, `pandas`, `pyarrow`, and `numpy`. This cell only installs missing packages, so it should not spend a long time reinstalling PyTorch.

In [ ]:
import importlib.util
import sys

missing = [pkg for pkg in ['torch', 'pandas', 'pyarrow', 'numpy'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    !{sys.executable} -m pip install -q {' '.join(missing)}
else:
    print('All required packages are available.')

import torch, pandas as pd, pyarrow, numpy as np
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('pandas:', pd.__version__)
print('pyarrow:', pyarrow.__version__)

## Config

For the first serious Colab run, this config uses a larger MLP than the local smoke test but still keeps the data prefix bounded so the run is manageable overnight.

In [ ]:
CONFIG = {
    'run_name': 'two_tower_baseline_colab_v1',
    'max_train_examples': 2_000_000,
    'max_validation_examples': 300_000,
    'target_classes': ['STRONG_POSITIVE'],
    'batch_size': 2048,
    'eval_batch_size': 2048,
    'epochs': 3,
    'learning_rate': 3e-4,
    'weight_decay': 1e-5,
    'retrieval_dim': 128,
    'user_hidden_dims': [256, 128],
    'item_hidden_dims': [256, 128],
    'dropout': 0.1,
    'temperature': 0.07,
    'top_ks': [10, 50, 100],
    'eval_max_batches': 100,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}
CONFIG

## Feature contract

Target/current-event columns are labels only. They are never passed to the model input.

In [ ]:
USER_CATEGORICAL = ('user_active_degree',)
ITEM_CATEGORICAL = ('video_type', 'upload_type')
USER_ID_FEATURES = ('user_id',)
ITEM_ID_FEATURES = ('video_id',)

USER_NUMERIC = (
    'is_lowactive_period', 'is_live_streamer', 'is_video_author',
    'follow_user_num', 'fans_user_num', 'friend_user_num', 'register_days',
    *[f'onehot_feat{i}' for i in range(18)],
    'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate',
    'user_hist_comment_rate', 'user_hist_forward_rate', 'user_hist_follow_rate',
    'user_hist_hate_rate', 'user_hist_avg_watch_ratio', 'user_hist_avg_play_time_sec',
    'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate',
    'session_prior_like_rate', 'session_prior_hate_rate', 'session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta',
    'event_hour', 'event_dayofweek', 'tab', 'is_rand',
)
ITEM_NUMERIC = (
    'video_duration_sec', 'aspect_ratio', 'visible_status', 'music_type',
    'upload_age_days_at_event', 'item_hist_events', 'item_hist_long_view_rate',
    'item_hist_like_rate', 'item_hist_hate_rate', 'item_hist_avg_watch_ratio',
)
TARGET_ONLY = {
    'target_class', 'is_positive', 'is_observed_negative', 'is_ambiguous',
    'engagement_strength', 'watch_ratio_clipped', 'long_view', 'is_like',
    'is_comment', 'is_forward', 'is_follow', 'is_hate', 'is_click',
    'is_profile_enter', 'play_time_ms', 'duration_ms', 'watch_ratio',
}
EMBED_DIMS = {
    'user_id': 8,
    'video_id': 16,
    'user_active_degree': 4,
    'video_type': 3,
    'upload_type': 8,
}

assert not (set(USER_NUMERIC) & TARGET_ONLY)
assert not (set(ITEM_NUMERIC) & TARGET_ONLY)
print('user numeric:', len(USER_NUMERIC))
print('item numeric:', len(ITEM_NUMERIC))

In [ ]:
import json
import random
from dataclasses import dataclass
from typing import Mapping

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class Vocabulary:
    feature_name: str
    value_to_index: dict
    oov_index: int = 0

    @property
    def size(self):
        return max(self.value_to_index.values(), default=0) + 1

    def encode_many(self, values):
        return [self.value_to_index.get(v, self.oov_index) if pd.notna(v) else self.oov_index for v in values]

    @classmethod
    def from_values(cls, feature_name, values):
        unique = sorted({v for v in values if pd.notna(v)})
        return cls(feature_name, {v: i + 1 for i, v in enumerate(unique)})


def parquet_files(path: Path):
    path = Path(path)
    if path.is_file():
        return [path] if path.name.endswith('.parquet') else []
    if not path.exists():
        raise FileNotFoundError(f'Missing path: {path}')
    files = sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files:
        visible = sorted(x.name for x in path.iterdir())[:30] if path.is_dir() else []
        raise FileNotFoundError(f'No parquet files found under {path}. Visible entries: {visible}')
    return files


def read_parquet_head(path: Path, max_rows: int | None, columns=None) -> pd.DataFrame:
    files = parquet_files(path)
    dataset = ds.dataset([str(f) for f in files], format='parquet')
    if columns is not None:
        missing = sorted(set(columns) - set(dataset.schema.names))
        if missing:
            raise ValueError(f'{path} is missing columns {missing}. Available columns: {dataset.schema.names}')
    if max_rows is None:
        return dataset.to_table(columns=columns).to_pandas()
    return dataset.head(max_rows, columns=columns).to_pandas()


def load_parquet_vocabulary(path: Path, feature_name: str) -> Vocabulary:
    df = read_parquet_head(path, None)
    return Vocabulary(feature_name, dict(zip(df[feature_name].tolist(), df[f'{feature_name}_idx'].astype(int).tolist())))


class NumericalPreprocessor:
    def __init__(self, stats: Mapping[str, Mapping[str, float]], clip=True, normalize=True, eps=1e-6):
        self.stats = stats
        self.clip = clip
        self.normalize = normalize
        self.eps = eps

    @classmethod
    def from_json(cls, path: Path):
        payload = json.loads(path.read_text())
        return cls(payload.get('features', payload))

    def transform_tensor(self, frame: pd.DataFrame, features: tuple[str, ...]) -> torch.Tensor:
        cols = []
        for feat in features:
            st = self.stats[feat]
            values = pd.to_numeric(frame[feat], errors='coerce').astype('float32')
            values = values.fillna(float(st.get('mean', 0.0) or 0.0))
            if self.clip:
                values = values.clip(float(st.get('p01', st.get('min', -np.inf))), float(st.get('p99', st.get('max', np.inf))))
            if self.normalize:
                mean = float(st.get('mean', 0.0) or 0.0)
                std = float(st.get('stddev', 0.0) or 0.0)
                values = (values - mean) / std if std > self.eps else values * 0.0
            cols.append(np.nan_to_num(values.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0))
        return torch.tensor(np.stack(cols, axis=1), dtype=torch.float32)


class BatchPreprocessor:
    def __init__(self, vocabularies, numerical):
        self.vocabularies = vocabularies
        self.numerical = numerical

    def encode_categorical(self, frame, features):
        return {f: torch.tensor(self.vocabularies[f].encode_many(frame[f].tolist()), dtype=torch.long) for f in features}

    def encode(self, frame):
        user_cat = self.encode_categorical(frame, USER_ID_FEATURES + USER_CATEGORICAL)
        item_cat = self.encode_categorical(frame, ITEM_ID_FEATURES + ITEM_CATEGORICAL)
        user_num = self.numerical.transform_tensor(frame, USER_NUMERIC)
        item_num = self.numerical.transform_tensor(frame, ITEM_NUMERIC)
        return user_cat, user_num, item_cat, item_num

## Load train/validation frames

Only `STRONG_POSITIVE` rows are used for this first retrieval baseline. `AMBIGUOUS_WEAK` and `OBSERVED_NEGATIVE` are held out for future loss/sampling designs.

In [ ]:
def load_joined_split(split: str, max_examples: int | None) -> pd.DataFrame:
    core_cols = ['example_id', 'user_id', 'video_id', 'as_of_time', 'split', 'is_warm_user', 'is_warm_item']
    target_cols = ['example_id', 'target_class', 'is_positive', 'is_observed_negative', 'is_ambiguous']
    user_cols = ['example_id', *USER_ID_FEATURES, *USER_CATEGORICAL, *USER_NUMERIC]
    item_cols = ['example_id', *ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC]

    core = read_parquet_head(GOLD_ROOT / split, max_examples, core_cols)
    targets = read_parquet_head(GOLD_ROOT / 'targets' / split, max_examples, target_cols)
    user_state = read_parquet_head(GOLD_ROOT / 'user_state' / split, max_examples, user_cols)
    item_features = read_parquet_head(GOLD_ROOT / 'item_features' / 'point_in_time' / split, max_examples, item_cols)

    frame = (
        core.merge(targets, on='example_id', how='inner')
        .merge(user_state, on=['example_id', 'user_id'], how='inner')
        .merge(item_features, on=['example_id', 'video_id'], how='inner')
    )
    frame = frame[frame['target_class'].isin(CONFIG['target_classes'])]
    return frame.reset_index(drop=True)

set_seed(CONFIG['seed'])
train_frame = load_joined_split('train', CONFIG['max_train_examples'])
validation_frame = load_joined_split('validation', CONFIG['max_validation_examples'])
print('train positives:', train_frame.shape)
print('validation positives:', validation_frame.shape)
display(train_frame.head(3))

In [ ]:
vocabularies = {
    'user_active_degree': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/user_active_degree', 'user_active_degree'),
    'video_type': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/video_type', 'video_type'),
    'upload_type': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/upload_type', 'upload_type'),
    'user_id': Vocabulary.from_values('user_id', train_frame['user_id'].tolist()),
    'video_id': Vocabulary.from_values('video_id', train_frame['video_id'].tolist()),
}
numerical = NumericalPreprocessor.from_json(GOLD_ROOT / 'transforms/numeric_stats.json')
preprocessor = BatchPreprocessor(vocabularies, numerical)

vocab_summary = {k: {'size': v.size, 'cardinality_excluding_oov': len(v.value_to_index)} for k, v in vocabularies.items()}
vocab_summary

In [ ]:
class TwoTowerFrameDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        return self.frame.iloc[idx].to_dict()


def collate(rows):
    frame = pd.DataFrame(rows)
    user_cat, user_num, item_cat, item_num = preprocessor.encode(frame)
    return user_cat, user_num, item_cat, item_num


class FeatureEncoder(nn.Module):
    def __init__(self, table_sizes: dict, numeric_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleDict({
            name: nn.Embedding(size, EMBED_DIMS[name], padding_idx=0)
            for name, size in table_sizes.items()
        })
        self.numeric_dim = numeric_dim
        self.output_dim = sum(EMBED_DIMS[name] for name in table_sizes) + numeric_dim

    def forward(self, categorical, numeric):
        pieces = [self.embeddings[name](categorical[name].long()) for name in self.embeddings]
        pieces.append(numeric.float())
        return torch.cat(pieces, dim=1)


def build_mlp(input_dim, hidden_dims, output_dim, dropout):
    layers = []
    prev = input_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
        prev = h
    layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)


class TwoTowerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_encoder = FeatureEncoder(
            {'user_id': vocabularies['user_id'].size, 'user_active_degree': vocabularies['user_active_degree'].size},
            len(USER_NUMERIC),
        )
        self.item_encoder = FeatureEncoder(
            {'video_id': vocabularies['video_id'].size, 'video_type': vocabularies['video_type'].size, 'upload_type': vocabularies['upload_type'].size},
            len(ITEM_NUMERIC),
        )
        self.user_tower = build_mlp(self.user_encoder.output_dim, CONFIG['user_hidden_dims'], CONFIG['retrieval_dim'], CONFIG['dropout'])
        self.item_tower = build_mlp(self.item_encoder.output_dim, CONFIG['item_hidden_dims'], CONFIG['retrieval_dim'], CONFIG['dropout'])

    def forward(self, user_cat, user_num, item_cat, item_num):
        user_vec = F.normalize(self.user_tower(self.user_encoder(user_cat, user_num)), dim=1)
        item_vec = F.normalize(self.item_tower(self.item_encoder(item_cat, item_num)), dim=1)
        logits = user_vec @ item_vec.T / CONFIG['temperature']
        return user_vec, item_vec, logits


def move_batch(batch, device):
    user_cat, user_num, item_cat, item_num = batch
    return (
        {k: v.to(device) for k, v in user_cat.items()},
        user_num.to(device),
        {k: v.to(device) for k, v in item_cat.items()},
        item_num.to(device),
    )

In [ ]:
def in_batch_metrics(logits, top_ks):
    n = logits.shape[0]
    labels = torch.arange(n, device=logits.device)
    ranks = (torch.argsort(logits, dim=1, descending=True) == labels[:, None]).nonzero()[:, 1] + 1
    metrics = {'mrr': torch.mean(1.0 / ranks.float()).item()}
    for raw_k in top_ks:
        k = min(raw_k, n)
        hits = ranks <= k
        metrics[f'hitrate@{raw_k}'] = hits.float().mean().item()
        metrics[f'recall@{raw_k}'] = hits.float().mean().item()
        ndcg = torch.where(hits, 1.0 / torch.log2(ranks.float() + 1.0), torch.zeros_like(ranks, dtype=torch.float32))
        metrics[f'ndcg@{raw_k}'] = ndcg.mean().item()
    return metrics


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    totals, examples = {}, 0
    for i, batch in enumerate(loader):
        if CONFIG['eval_max_batches'] is not None and i >= CONFIG['eval_max_batches']:
            break
        batch = move_batch(batch, device)
        _, _, logits = model(*batch)
        m = in_batch_metrics(logits, CONFIG['top_ks'])
        bs = logits.shape[0]
        examples += bs
        for k, v in m.items():
            totals[k] = totals.get(k, 0.0) + v * bs
    return {k: v / max(examples, 1) for k, v in totals.items()} | {'examples': examples}

## Train

This saves artifacts directly to Drive. If Colab disconnects after training, check `OUTPUT_DIR` for `metrics.json`, `comparison_summary.json`, and `model.pt`.

In [ ]:
device = torch.device(CONFIG['device'])
model = TwoTowerModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(TwoTowerFrameDataset(train_frame), batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate, num_workers=0)
validation_loader = DataLoader(TwoTowerFrameDataset(validation_frame), batch_size=CONFIG['eval_batch_size'], shuffle=False, collate_fn=collate, num_workers=0)

history = []
for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    total_loss, total_examples = 0.0, 0
    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch, device)
        _, _, logits = model(*batch)
        labels = torch.arange(logits.shape[0], device=device)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        bs = logits.shape[0]
        total_loss += loss.item() * bs
        total_examples += bs
        if step % 100 == 0:
            print(f'epoch={epoch} step={step} loss={loss.item():.4f}')
    val = evaluate(model, validation_loader, device)
    row = {'epoch': epoch, 'train_loss': total_loss / max(total_examples, 1), 'train_examples': total_examples, **{f'validation_{k}': v for k, v in val.items()}}
    history.append(row)
    print(row)

final_validation = evaluate(model, validation_loader, device)
print('Final validation:', final_validation)

In [ ]:
baseline_summary = {}
if ALS_SUMMARY_PATH.exists():
    als = json.loads(ALS_SUMMARY_PATH.read_text())
    baseline_summary = {
        'popularity_test': als.get('popularity_test'),
        'als_test': als.get('final_als_test'),
        'selected_als_hyperparameters': als.get('selected_als_hyperparameters'),
    }

metrics = {
    'run_name': CONFIG['run_name'],
    'device': str(device),
    'train_rows_loaded': len(train_frame),
    'validation_rows_loaded': len(validation_frame),
    'history': history,
    'final_validation': final_validation,
    'model_parameter_count': sum(p.numel() for p in model.parameters()),
    'vocab_summary': vocab_summary,
    'baseline_summary': baseline_summary,
    'note': 'Two-Tower metrics are in-batch retrieval metrics; ALS/popularity use logged top-K evaluation, so compare directionally only.',
}

(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True, default=str))
(OUTPUT_DIR / 'comparison_summary.json').write_text(json.dumps({
    'two_tower_validation': final_validation,
    'baseline_summary': baseline_summary,
    'note': metrics['note'],
}, indent=2, sort_keys=True, default=str))
(OUTPUT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2, sort_keys=True, default=str))
(OUTPUT_DIR / 'vocab_summary.json').write_text(json.dumps(vocab_summary, indent=2, sort_keys=True, default=str))
torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG, 'vocab_summary': vocab_summary}, OUTPUT_DIR / 'model.pt')

print('Saved artifacts to:', OUTPUT_DIR)
print((OUTPUT_DIR / 'metrics.json').read_text()[:3000])